# Manifold Boundary Approximation for Multi-Body Rigid-Body Lattice Models

MBAM (Transtrum & Qiu 2014) for the models of `rigid_body_diffuse_general.ipynb`: any space
group, any number of bodies per asymmetric unit, with internal springs between bodies cut from
one chain. It follows `mbam_6o2h.ipynb` in its coordinates, its family of reduced models and its
loop, with the forward model generalized from one body per cell to many.

**Input.** `OUT_PREFIX + 'mbam_bundle.npz'`, written by the last cells of the general notebook.

**Coordinates and reduced models (as in `mbam_6o2h.ipynb`).** Each orbit's local stiffness is
$K=K_{\rm prior}^{1/2}U_Fe^{S_F}U_F^{\mathsf T}K_{\rm prior}^{1/2}$ in a frame $R=R_0e^{\Xi}$ of
prior-whitened space split into constrained (**C**, infinite stiffness), released (**Z**, zero) and
finite (**F**) directions; the coordinates are the symmetric $S_F$ ($\phi$: diagonal and
$\sqrt2\times$ upper entries) and the cross-block angles of $\Xi$. The full model is all **F**, and
then $\|\phi\|^2$ is exactly the geodesic distance to the prior. A limit sends one eigenvalue of
$S_F$ to $+\infty$ (**C**) or $-\infty$ (**Z**) and removes exactly one parameter.

**Infinite stiffness, exactly.** A constrained direction $c$ of member $m$ is the linear
constraint $C^\dagger u=0$ with $C=B_m(\mathbf q)^\dagger c$, where $B_m(\mathbf q)u$ is the member's
relative displacement at its contact point; with several bodies
$B_m=T_aE_a-e^{i\mathbf q\cdot\mathbf R}T_bE_b$. The intensity is $I=G^\dagger v$ from the bordered
system $\begin{pmatrix}D&C\\C^\dagger&0\end{pmatrix}\begin{pmatrix}v\\\mu\end{pmatrix}=\begin{pmatrix}G\\0\end{pmatrix}$,
with closed-form derivatives $\partial I/\partial K=-\mathrm{Re}(w^*w^{\mathsf T})$ and
$\partial I/\partial c=-2\,\mathrm{Re}(\mu\,w^*)$, $w=B_mv$. Constraint columns that vanish at a
point are dropped there. Welding two bodies is six **C** limits of the spring between them.

**The loop.** Start from the bundle's (regularized) fit, refit without the prior
(`MBAM_START = 'unregularized'`) behind a soft wall at $|\sigma|=$ `S_BARRIER`, and take every
eigenvalue the fit pushes to the wall as a limit at once (*fit-driven*). Each step then:

1. computes the Fisher spectrum $g$ of $J^{\mathsf T}J$ and the sloppiest directions, from the full
   spectrum and from the stiffness block with the angles relaxed (Schur complement);
2. if the sloppiest $g<$ `G_DIRECT`, sends its dominant eigenvalue straight to its limit and
   accepts if that changes the predictions by less than `LIMIT_TOL` in $\chi^2$ (*direct*);
3. otherwise integrates the geodesic $\ddot\phi=-(J^{\mathsf T}J)^+J^{\mathsf T}A_{vv}$ at unit speed in
   data space from each starting direction, both ways, and stops as soon as an eigenvalue that
   is far out ($|\sigma|\ge$ `S_ID_MIN`), moving outward and carrying at least `SHARE_MIN` of the
   squared speed can be sent to its limit for less than `LIMIT_TOL` (*geodesic*);
4. refits the reduced model; among the boundaries found, `BOUNDARY_CHOICE = 'cheapest'` keeps the
   one with the lowest refitted $\chi^2$, `'closest'` the shortest arc length.

**Simplifications relative to `mbam_6o2h.ipynb`:** one model is followed (no beam search), there
is no greedy continuation and no freezing of angles, limits are single eigenvalues, and the
class, movie and dashboard sections are left out. The report focuses on which orbits, and
which internal springs, reach their limits, in what order and at what cost.

In [ ]:
import time
import numpy as np
import scipy.linalg as sla
from scipy.optimize import least_squares
from scipy.integrate import RK45
import matplotlib.pyplot as plt

BUNDLE = 'synthetic_6o2h_gnm_rigid_mbam_bundle.npz'   # written by rigid_body_diffuse_general.ipynb
OUT    = 'mbam_mb_'                                    # prefix of every figure written here

MBAM_START      = 'unregularized'   # refit the full model with lambda = 0 first ('regularized': keep the fit)
MBAM_MAX_STEPS  = 30                # 0: stop after the welding test (section 5)
MBAM_TIME_BUDGET = None             # seconds; the loop stops cleanly after this long
TAU_MAX         = 100.0             # geodesic arc-length budget (sigma units, ~ sqrt(delta chi^2))
S_CAP           = 15.0              # geodesic stops when a log-stiffness eigenvalue reaches this
SPEED_CAP       = 1e4               # ... or when the parameter speed grows by this factor
S_ID_MIN        = 4.0               # an eigenvalue must be this far out to count as a limit
SHARE_MIN       = 0.05              # and carry at least this share of the squared speed
LIMIT_TOL       = 0.5               # chi^2 change allowed when the limit is taken (no refit)
G_DIRECT        = 1e-6              # below this g, try the limit of the sloppiest direction directly
N_TRY_DIRS      = 2                 # starting directions per family (full spectrum, stiffness block)
BOUNDARY_CHOICE = 'cheapest'        # 'cheapest' (lowest refitted chi^2) or 'closest' (shortest arc)
GEO_RTOL, GEO_ATOL = 1e-4, 1e-6     # geodesic ODE tolerances
GEO_MAX_RHS     = 1500              # right-hand-side evaluations per geodesic
H_AVV           = 1e-3              # step of the second difference that gives A_vv (unit data speed)
S_BARRIER       = 17.0              # soft wall on |log-stiffness eigenvalue| during fits
BARRIER_W       = 1.0
RUNAWAY_MARGIN  = 0.5               # an eigenvalue this close to the wall after a fit is a limit
FIT_MAX_NFEV    = 200
FIT_ROUNDS      = 3
WELD_TEST       = True              # cost of welding every internal spring (section 5)

## 1. The bundle

In [ ]:
Z = np.load(BUNDLE, allow_pickle=False)
NBODY = int(Z['n_body']); NDOF = 6*NBODY
B_RECIP = Z['B_recip']
ORB = [str(s) for s in Z['orbit_labels']]; NO = len(ORB)
INTERNAL = Z['orbit_internal'].astype(bool)
MA, MB, MO = Z['mem_a'], Z['mem_b'], Z['mem_orbit']
MRN, MTA, MTB, MQ, MNQ = Z['mem_Rn'], Z['mem_Ta'], Z['mem_Tb'], Z['mem_Q'], Z['mem_nQ']
NM = len(MA)
K_PRIOR, K_FIT = Z['K_prior_local'], Z['K_fit_local']
SYN = 'K_true_local' in Z.files
K_TRUE = Z['K_true_local'] if SYN else None

def sym_sqrt(A, inv=False):
    w, V = np.linalg.eigh(0.5*(A + A.T))
    w = np.maximum(w, 1e-300)
    return (V*w**(-0.5 if inv else 0.5)) @ V.T
WH  = np.array([sym_sqrt(K) for K in K_PRIOR])            # K_prior^{1/2}
WHI = np.array([sym_sqrt(K, inv=True) for K in K_PRIOR])

USE_BG = bool(Z['use_background']); BG_KNOTS = Z['bg_knots']
def _bg_basis(qn):
    x = np.clip(qn, BG_KNOTS[0], BG_KNOTS[-1])
    j = np.clip(np.searchsorted(BG_KNOTS, x, side='right') - 1, 0, len(BG_KNOTS) - 2)
    t = (x - BG_KNOTS[j])/(BG_KNOTS[j + 1] - BG_KNOTS[j])
    Bm = np.zeros((len(x), len(BG_KNOTS))); ar = np.arange(len(x))
    Bm[ar, j] = 1 - t; Bm[ar, j + 1] += t
    return Bm

class DataSet:
    def __init__(self, q, G, y, w, name):
        self.q, self.G, self.y, self.w, self.name = np.asarray(q, float), np.asarray(G), np.asarray(y, float), np.asarray(w, float), name
        self.ph = np.exp(1j*((self.q @ B_RECIP.T) @ MRN.T))       # (N, members)
        self.sw = np.sqrt(self.w)
        self.U = None
        if USE_BG:   # the profiled isotropic background: project it out
            U, s, _ = np.linalg.svd(self.sw[:, None]*_bg_basis(np.linalg.norm(self.q @ B_RECIP.T, axis=1)), full_matrices=False)
            self.U = U[:, s > 1e-10*s.max()]
    def __len__(self):
        return len(self.y)

TRAIN = DataSet(Z['q_fit'], Z['G_fit'], Z['I_fit'], Z['w_fit'], 'train')
TEST  = DataSet(Z['q_test'], Z['G_test'], Z['I_test'], 1.0/Z['sigma_test']**2, 'held out')
print(f"{BUNDLE}: {str(Z['pdb_id'])} ({str(Z['sg_hm'])}), bodies by {str(Z['body_method'])!r}: {NBODY} bodies, {NM} member "
      f"bonds in {NO} orbits; data {str(Z['data_source'])}: {len(TRAIN)} training and {len(TEST)} held-out points")
print("internal orbits: " + (', '.join(o for o, i in zip(ORB, INTERNAL) if i) or 'none'))
if SYN:
    print(f"synthetic truth: SYN_INTERNAL = {str(Z['syn_internal'])!r} ({float(Z['syn_f_int']):g} x the prior)")

## 2. Coordinates, reduced models and the forward model

In [ ]:
SQ2 = np.sqrt(2.0)
def _sym_basis(n):
    B = []
    for i in range(n):
        E = np.zeros((n, n)); E[i, i] = 1.0; B.append(E)
    for i in range(n):
        for j in range(i + 1, n):
            E = np.zeros((n, n)); E[i, j] = E[j, i] = 1/SQ2; B.append(E)
    return np.array(B, dtype=float).reshape(len(B), n, n)
SB = {n: _sym_basis(n) for n in range(7)}
def sym_vec(S):
    return np.einsum('kij,ij->k', SB[len(S)], S)
def sym_mat(s, n):
    return np.einsum('k,kij->ij', np.asarray(s, float), SB[n])

class Model:
    """Per orbit: a frame R0 of whitened space and a C/Z/F label per column."""
    def __init__(self, R0, lab):
        self.R0 = [np.array(r, float) for r in R0]
        self.lab = [np.array(list(l)) for l in lab]
        self.lay, off = [], 0
        for L in self.lab:
            F, C, Zz = (np.flatnonzero(L == k) for k in 'FCZ')
            pairs = [(i, j) for i in range(6) for j in range(i + 1, 6) if L[i] != L[j]]
            nF = len(F); ns = nF*(nF + 1)//2
            self.lay.append(dict(F=F, C=C, Z=Zz, pairs=pairs, nF=nF, ns=ns, off=off, n=len(pairs) + ns))
            off += len(pairs) + ns
        self.p = off
    def counts(self, o):
        L = self.lay[o]
        return len(L['C']), len(L['Z']), L['nF']

def full_model():
    return Model([np.eye(6)]*NO, [['F']*6]*NO)

def phi_from_K(Kloc):
    """Full-model coordinates of a set of local stiffness matrices."""
    xs = []
    for o in range(NO):
        P = WHI[o] @ Kloc[o] @ WHI[o]
        w, V = np.linalg.eigh(0.5*(P + P.T))
        xs.append(sym_vec((V*np.log(np.maximum(w, 1e-300))) @ V.T))
    return np.concatenate(xs)

def unpack(model, x):
    out = []
    for L in model.lay:
        xo = x[L['off']:L['off'] + L['n']]
        Xi = np.zeros((6, 6))
        for k, (i, j) in enumerate(L['pairs']):
            Xi[i, j], Xi[j, i] = xo[k], -xo[k]
        out.append((Xi, sym_mat(xo[len(L['pairs']):], L['nF'])))
    return out

def orbit_state(model, o, Xi, SF):
    L = model.lay[o]
    U = model.R0[o] @ (sla.expm(Xi) if np.any(Xi) else np.eye(6))
    if L['nF']:
        lam, V = np.linalg.eigh(0.5*(SF + SF.T)); eS = (V*np.exp(lam)) @ V.T
    else:
        lam, V, eS = np.zeros(0), np.zeros((0, 0)), np.zeros((0, 0))
    UF, UC = U[:, L['F']], U[:, L['C']]
    return dict(U=U, UF=UF, UC=UC, eS=eS, lam=lam, V=V, K=WH[o] @ UF @ eS @ UF.T @ WH[o], C=WH[o] @ UC)

def forward(model, x, ds, need_jac=False):
    """I(q) (and dI/dx) for the reduced model at x, with exact constraints (bordered system)."""
    up = unpack(model, x)
    ost = [orbit_state(model, o, *up[o]) for o in range(NO)]
    N = len(ds)
    C0 = np.zeros((NDOF, NDOF)); D = np.zeros((N, NDOF, NDOF), complex)
    Clab, ncol = [], []
    for m in range(NM):
        o = MO[m]; Qs = MQ[m, :MNQ[m]]
        K = sum(Q @ ost[o]['K'] @ Q.T for Q in Qs)/len(Qs)
        nc_o = ost[o]['C'].shape[1]
        if nc_o:
            assert len(Qs) == 1, "constraints on orbits with a non-trivial stabilizer are not supported"
        Clab.append(Qs[0] @ ost[o]['C'] if nc_o else np.zeros((6, 0))); ncol.append(nc_o)
        a, b, Ta, Tb = MA[m], MB[m], MTA[m], MTB[m]
        sa, sb = slice(6*a, 6*a + 6), slice(6*b, 6*b + 6)
        C0[sa, sa] += Ta.T @ K @ Ta; C0[sb, sb] += Tb.T @ K @ Tb
        Bab = Ta.T @ K @ Tb
        D[:, sa, sb] -= ds.ph[:, m, None, None]*Bab
        D[:, sb, sa] -= np.conj(ds.ph[:, m])[:, None, None]*Bab.T
    D += 0.5*(C0 + C0.T)[None]
    nc = sum(ncol)
    if nc:
        Cc = np.zeros((N, NDOF, nc), complex); k0 = 0
        for m in range(NM):
            for k in range(ncol[m]):
                c = Clab[m][:, k]
                Cc[:, 6*MA[m]:6*MA[m] + 6, k0] += (MTA[m].T @ c)[None]
                Cc[:, 6*MB[m]:6*MB[m] + 6, k0] -= np.conj(ds.ph[:, m])[:, None]*(MTB[m].T @ c)[None]
                k0 += 1
        nrm = np.linalg.norm(Cc, axis=1)
        drop = nrm < 1e-9*np.maximum(nrm.max(0, keepdims=True), 1e-300)
        Cc = np.where(drop[:, None, :], 0.0, Cc)
        A = np.zeros((N, NDOF + nc, NDOF + nc), complex)
        A[:, :NDOF, :NDOF] = D; A[:, :NDOF, NDOF:] = Cc; A[:, NDOF:, :NDOF] = np.conj(np.swapaxes(Cc, 1, 2))
        A[:, NDOF + np.arange(nc), NDOF + np.arange(nc)] = drop
        sol = np.linalg.solve(A, np.concatenate([ds.G, np.zeros((N, nc))], axis=1)[:, :, None])[:, :, 0]
        v, mu = sol[:, :NDOF], sol[:, NDOF:]
    else:
        v = np.linalg.solve(D, ds.G[:, :, None])[:, :, 0]; mu = np.zeros((N, 0))
    I = np.real(np.sum(np.conj(ds.G)*v, axis=1))
    if not need_jac:
        return I
    GK = [np.zeros((N, 6, 6)) for _ in range(NO)]
    GC = [np.zeros((N, 6, ost[o]['C'].shape[1])) for o in range(NO)]
    k0 = 0
    for m in range(NM):
        o = MO[m]; Qs = MQ[m, :MNQ[m]]
        w = v[:, 6*MA[m]:6*MA[m] + 6] @ MTA[m].T - ds.ph[:, m, None]*(v[:, 6*MB[m]:6*MB[m] + 6] @ MTB[m].T)
        S = -np.real(w[:, :, None]*np.conj(w)[:, None, :])               # dI/dK (lab, at the contact)
        for Q in Qs:
            GK[o] += np.einsum('ia,nij,jb->nab', Q, S, Q)/len(Qs)
        for k in range(ncol[m]):
            GC[o][:, :, k] += (-2*np.real(mu[:, k0][:, None]*np.conj(w))) @ Qs[0]   # dI/dc, local
            k0 += 1
    Jm = np.zeros((N, model.p))
    for o, L in enumerate(model.lay):
        st, (Xi, SF), W = ost[o], up[o], WH[o]
        npair = len(L['pairs'])
        if L['nF']:
            lam, V = st['lam'], st['V']
            el, dl = np.exp(lam), lam[:, None] - lam[None, :]
            same = np.abs(dl) < 1e-10
            Phi = np.where(same, el[:, None], (el[:, None] - el[None, :])/np.where(same, 1.0, dl))
            Mk = np.einsum('ai,kab,bj->kij', V, SB[L['nF']], V)
            dE = np.einsum('ia,kab,jb->kij', V, Phi[None]*Mk, V)        # Frechet derivative of exp
            Y = W @ st['UF'] @ dE @ st['UF'].T @ W
            Jm[:, L['off'] + npair:L['off'] + L['n']] = np.einsum('nab,kab->nk', GK[o], Y)
        for kk, (i, j) in enumerate(L['pairs']):
            E = np.zeros((6, 6)); E[i, j], E[j, i] = 1.0, -1.0
            dU = model.R0[o] @ (sla.expm_frechet(Xi, E, compute_expm=False) if np.any(Xi) else E)
            dUF, dUC = dU[:, L['F']], dU[:, L['C']]
            col = np.einsum('nab,ab->n', GK[o], W @ (dUF @ st['eS'] @ st['UF'].T + st['UF'] @ st['eS'] @ dUF.T) @ W)
            if len(L['C']):
                col = col + np.einsum('nik,ik->n', GC[o], W @ dUC)
            Jm[:, L['off'] + kk] = col
    return I, Jm

def resid(model, x, ds):
    r = ds.sw*(forward(model, x, ds) - ds.y)
    return r - ds.U @ (ds.U.T @ r) if ds.U is not None else r

def resid_jac(model, x, ds):
    I, Jm = forward(model, x, ds, True)
    r, J = ds.sw*(I - ds.y), ds.sw[:, None]*Jm
    if ds.U is not None:
        r = r - ds.U @ (ds.U.T @ r); J = J - ds.U @ (ds.U.T @ J)
    return r, J

def chi2(model, x, ds):
    return float(np.sum(resid(model, x, ds)**2))

def barrier(model, x):
    up = unpack(model, x); rb, Jb = [], []
    for o, L in enumerate(model.lay):
        if not L['nF']:
            continue
        lam, V = np.linalg.eigh(up[o][1])
        for j in range(L['nF']):
            ex = abs(lam[j]) - S_BARRIER
            row = np.zeros(model.p)
            if ex > 0:
                row[L['off'] + len(L['pairs']):L['off'] + L['n']] = BARRIER_W*np.sign(lam[j])*np.einsum(
                    'a,kab,b->k', V[:, j], SB[L['nF']], V[:, j])
            rb.append(BARRIER_W*max(ex, 0.0)); Jb.append(row)
    return np.array(rb), np.array(Jb).reshape(-1, model.p)

def fit_model(model, x0, ds=None, max_nfev=None, rounds=None):
    """Unregularized least squares behind the soft wall on |sigma|."""
    ds = TRAIN if ds is None else ds
    x = np.array(x0, float)
    if model.p == 0:
        return x
    n_res = len(ds) + sum(L['nF'] for L in model.lay)
    def f(z):
        """Residuals and the wall; NaN (the trust region then shrinks) where a trial step
        overflows or makes the bordered system singular."""
        try:
            with np.errstate(all='ignore'):
                r = np.r_[resid(model, z, ds), barrier(model, z)[0]]
        except (np.linalg.LinAlgError, ValueError, FloatingPointError):
            return np.full(n_res, np.nan)
        return r if np.all(np.isfinite(r)) else np.full(n_res, np.nan)
    jf = lambda z: np.vstack([resid_jac(model, z, ds)[1], barrier(model, z)[1]])
    for _ in range(rounds or FIT_ROUNDS):
        c0 = 0.5*float(np.sum(f(x)**2))
        res = least_squares(f, x, jac=jf, method='trf', x_scale='jac', max_nfev=max_nfev or FIT_MAX_NFEV,
                            ftol=1e-10, xtol=1e-12, gtol=1e-10)
        x = res.x
        if res.status != 0 or res.cost > c0 - 1e-6:
            break
    return x

## 3. Limits, geodesics and starting directions

In [ ]:
def spectral(model, x, v):
    """Eigenvalues sigma of every orbit's S_F, their velocities along v, and their share of the
    squared parameter speed."""
    up = unpack(model, x); out = []
    vn2 = float(v @ v)
    for o, L in enumerate(model.lay):
        if not L['nF']:
            continue
        lam, V = np.linalg.eigh(up[o][1])
        dS = sym_mat(v[L['off'] + len(L['pairs']):L['off'] + L['n']], L['nF'])
        dlam = np.einsum('aj,ab,bj->j', V, dS, V)
        for j in range(L['nF']):
            out.append(dict(o=o, j=j, sigma=float(lam[j]), dsigma=float(dlam[j]),
                            share=float(dlam[j]**2/max(vn2, 1e-300))))
    return out

def candidates(model, x, v):
    c = [d for d in spectral(model, x, v)
         if abs(d['sigma']) >= S_ID_MIN and d['sigma']*d['dsigma'] > 0 and d['share'] >= SHARE_MIN]
    return sorted(c, key=lambda d: -d['share'])

def apply_limit(model, x, o, j, kind):
    """Send eigenvalue j of orbit o's S_F to +inf (kind 'C') or -inf ('Z'). The frame is
    rebased on the current eigenvectors; every other parameter keeps its value."""
    up = unpack(model, x); Xi, SF = up[o]; L = model.lay[o]
    U = model.R0[o] @ (sla.expm(Xi) if np.any(Xi) else np.eye(6))
    lam, V = np.linalg.eigh(SF)
    UFr = U[:, L['F']] @ V
    keep = [i for i in range(L['nF']) if i != j]
    Cc = [U[:, L['C']]] + ([UFr[:, [j]]] if kind == 'C' else [])
    Zc = [U[:, L['Z']]] + ([UFr[:, [j]]] if kind == 'Z' else [])
    nC, nZ = sum(c.shape[1] for c in Cc), sum(c.shape[1] for c in Zc)
    R0s, labs = list(model.R0), list(model.lab)
    R0s[o] = np.hstack(Cc + Zc + [UFr[:, keep]]); labs[o] = ['C']*nC + ['Z']*nZ + ['F']*len(keep)
    m2 = Model(R0s, labs)
    xs = []
    for q, L2 in enumerate(m2.lay):
        if q == o:
            xo = np.zeros(L2['n'])
            if keep:
                xo[len(L2['pairs']):] = sym_vec(np.diag(lam[keep]))
        else:
            Lq = model.lay[q]; xo = x[Lq['off']:Lq['off'] + Lq['n']]
        xs.append(xo)
    return m2, np.concatenate(xs)

def fit_driven(model, x):
    """Take every eigenvalue the fit pushed to the wall as a limit, refit, repeat."""
    taken = []
    while True:
        run = [d for d in spectral(model, x, np.zeros(model.p)) if abs(d['sigma']) > S_BARRIER - RUNAWAY_MARGIN]
        if not run:
            return model, x, taken
        while run:          # all of them, then one refit
            d = max(run, key=lambda d: abs(d['sigma']))
            kind = 'C' if d['sigma'] > 0 else 'Z'
            model, x = apply_limit(model, x, d['o'], d['j'], kind)
            taken.append((d['o'], kind, d['sigma']))
            run = [d for d in spectral(model, x, np.zeros(model.p)) if abs(d['sigma']) > S_BARRIER - RUNAWAY_MARGIN]
        x = fit_model(model, x)

def geodesic(model, x0, v0, ds, tau_max):
    """Unit-speed geodesic from x0 along v0; stops at the first limit that costs < LIMIT_TOL."""
    P, cnt = model.p, [0]
    _, J0 = resid_jac(model, x0, ds)
    v0 = v0/max(np.linalg.norm(J0 @ v0), 1e-300)
    def rhs(t, y):
        cnt[0] += 1
        x, v = y[:P], y[P:]
        r, J = resid_jac(model, x, ds)
        A = (resid(model, x + H_AVV*v, ds) - 2*r + resid(model, x - H_AVV*v, ds))/H_AVV**2
        return np.r_[v, -np.linalg.lstsq(J, A, rcond=None)[0]]
    sol = RK45(rhs, 0.0, np.r_[x0, v0], tau_max, rtol=GEO_RTOL, atol=GEO_ATOL)
    v_ref = np.linalg.norm(v0)
    while sol.status == 'running':
        try:
            sol.step()
        except np.linalg.LinAlgError:
            return dict(found=False, reason='singular', tau=sol.t, nrhs=cnt[0])
        if sol.status == 'failed':
            return dict(found=False, reason='ode failed', tau=sol.t, nrhs=cnt[0])
        x, v = sol.y[:P], sol.y[P:]
        sp = spectral(model, x, v)
        capped = (any(abs(d['sigma']) > S_CAP for d in sp) or np.linalg.norm(v) > SPEED_CAP*v_ref
                  or cnt[0] > GEO_MAX_RHS)
        cands = candidates(model, x, v)
        if cands:
            r_x = resid(model, x, ds)
            for d in cands[:2]:
                try:
                    m2, x2 = apply_limit(model, x, d['o'], d['j'], 'C' if d['sigma'] > 0 else 'Z')
                    jump = float(np.sum((resid(m2, x2, ds) - r_x)**2))
                except np.linalg.LinAlgError:
                    continue
                if jump < LIMIT_TOL:
                    return dict(found=True, tau=float(sol.t), limit=d, model=m2, x=x2, jump=jump, nrhs=cnt[0])
        if capped:
            return dict(found=False, reason='cap', tau=float(sol.t), nrhs=cnt[0])
    return dict(found=False, reason='tau_max', tau=float(sol.t), nrhs=cnt[0])

def start_directions(model, x, ds):
    """Sloppiest directions of J^T J, and of its stiffness block with the angles relaxed."""
    _, J = resid_jac(model, x, ds); F = J.T @ J
    g, V = np.linalg.eigh(0.5*(F + F.T))
    dirs = [(float(g[k]), V[:, k], 'full') for k in range(min(N_TRY_DIRS, len(g)))]
    sidx = np.concatenate([np.arange(L['off'] + len(L['pairs']), L['off'] + L['n']) for L in model.lay]).astype(int)
    oidx = np.setdiff1d(np.arange(model.p), sidx)
    if len(oidx) and len(sidx):
        Foo_p = np.linalg.pinv(F[np.ix_(oidx, oidx)]); Fso = F[np.ix_(sidx, oidx)]
        Sc = F[np.ix_(sidx, sidx)] - Fso @ Foo_p @ Fso.T
        gs, Vs = np.linalg.eigh(0.5*(Sc + Sc.T))
        for k in range(min(N_TRY_DIRS, len(gs))):
            vv = np.zeros(model.p); vv[sidx] = Vs[:, k]; vv[oidx] = -Foo_p @ Fso.T @ Vs[:, k]
            dirs.append((float(gs[k]), vv/np.linalg.norm(vv), 'stiffness'))
    return dirs, g

def describe(model, o):
    nC, nZ, nF = model.counts(o)
    return f"{ORB[o]}{' (internal)' if INTERNAL[o] else ''}: C{nC} Z{nZ} F{nF}"

### Checks

The analytic Jacobian against central differences, for the full model and for a reduced model with constrained and released directions and free angles; and the exact weld against a very stiff spring.

In [ ]:
# --- Checks of the forward model ---------------------------------------------------------------------
M_FULL = full_model()
PHI_REG = phi_from_K(K_FIT)
_ds = DataSet(TRAIN.q[:40], TRAIN.G[:40], TRAIN.y[:40], TRAIN.w[:40], 'check')
def _jac_check(model, x0, n=6, h=1e-5):
    _, Jm = forward(model, x0, _ds, True)
    errs = []
    for k in np.random.default_rng(0).choice(model.p, min(n, model.p), replace=False):
        xp, xm = x0.copy(), x0.copy(); xp[k] += h; xm[k] -= h
        fd = (forward(model, xp, _ds) - forward(model, xm, _ds))/(2*h)
        errs.append(np.linalg.norm(fd - Jm[:, k])/max(np.linalg.norm(fd), 1e-300))
    return max(errs)
_e1 = _jac_check(M_FULL, PHI_REG)
print(f"Jacobian vs central differences, full model: max relative error {_e1:.1e}")
_o = int(np.argmax(INTERNAL)) if INTERNAL.any() else 0
_m2, _x2 = M_FULL, PHI_REG.copy()
for kind in ('C', 'Z', 'C'):
    _m2, _x2 = apply_limit(_m2, _x2, _o, 0, kind)
_x2[_m2.lay[_o]['off']:_m2.lay[_o]['off'] + len(_m2.lay[_o]['pairs'])] = 0.05     # non-zero angles
_e2 = _jac_check(_m2, _x2, n=10)
print(f"Jacobian vs central differences, orbit {ORB[_o]} with 2 constrained, 1 released direction and "
      f"non-zero angles: max relative error {_e2:.1e}")
assert max(_e1, _e2) < 1e-4, "analytic and finite-difference Jacobians disagree"
# the exact weld (all six directions of an orbit constrained) against a 10^6 x stiffer spring
_mw, _xw = M_FULL, PHI_REG.copy()
for _ in range(6):
    _mw, _xw = apply_limit(_mw, _xw, _o, 0, 'C')
_xs = PHI_REG.copy(); _L = M_FULL.lay[_o]
_S = sym_mat(_xs[_L['off']:_L['off'] + _L['n']], 6) + np.log(1e6)*np.eye(6)
_xs[_L['off']:_L['off'] + _L['n']] = sym_vec(_S)
_Iw, _Is = forward(_mw, _xw, _ds), forward(M_FULL, _xs, _ds)
print(f"orbit {ORB[_o]} welded exactly vs 10^6 x stiffer: max relative difference {np.max(np.abs(_Iw - _Is)/np.abs(_Is)):.1e}")
assert np.max(np.abs(_Iw - _Is)/np.abs(_Is)) < 1e-3

## 4. The start point

In [ ]:
# --- Start point ------------------------------------------------------------------------------------
_t0 = time.time()
def log_sigmas(Kloc, o):
    P = WHI[o] @ Kloc[o] @ WHI[o]
    return np.sort(np.log(np.maximum(np.linalg.eigvalsh(0.5*(P + P.T)), 1e-300)))
model, x = full_model(), phi_from_K(K_FIT)
CHI2_REG = (chi2(model, x, TRAIN), chi2(model, x, TEST))
if MBAM_START == 'unregularized':
    x = fit_model(model, x, rounds=FIT_ROUNDS + 2)
model, x, _taken0 = fit_driven(model, x)
records = [dict(step=0, action='start', limits=[(o, k, s) for o, k, s in _taken0], p=model.p,
                chi2=chi2(model, x, TRAIN), chi2_test=chi2(model, x, TEST), tau=0.0,
                status={o: model.counts(o) for o in range(NO)}, model=model, x=x)]
print(f"regularized fit from the bundle: chi^2 train {CHI2_REG[0]:.1f}, held out {CHI2_REG[1]:.1f} "
      f"({len(TRAIN)} / {len(TEST)} points)")
print(f"after the {MBAM_START} start: chi^2 train {records[0]['chi2']:.1f}, held out {records[0]['chi2_test']:.1f}, "
      f"{model.p} parameters ({time.time() - _t0:.0f} s)")
for o in np.flatnonzero(INTERNAL):
    print(f"internal orbit {ORB[o]}, log-eigenvalues vs the prior:")
    print(f"   regularized fit {np.round(log_sigmas(K_FIT, o), 2)}")
    _K_now = [orbit_state(model, q, *unpack(model, x)[q])['K'] for q in range(NO)]
    if model.counts(o)[2] == 6:
        print(f"   {MBAM_START} fit {np.round(log_sigmas(_K_now, o), 2)}")
    if SYN:
        print(f"   truth           {np.round(log_sigmas(K_TRUE, o), 2)}")
if _taken0:
    print("fit-driven limits at the start: " + ", ".join(f"{ORB[o]} -> {'inf' if k == 'C' else '0'} (sigma {s:.1f})"
                                                       for o, k, s in _taken0))

## 5. Welding test

For every internal orbit (a spring between two bodies cut from one chain): where its stiffness
eigenvalues sit after the start, where its directions rank in the Fisher spectrum, what sending each
finite eigenvalue to infinite or to zero stiffness costs in $\chi^2$ without a refit (MBAM's direct
test; `LIMIT_TOL` is the threshold for taking a limit at once), and the cost of the full weld (every
finite direction sent to infinity) after a refit, with $\Delta$AIC. A weld that costs little after
refitting means the data cannot tell the two bodies from one. With `MBAM_MAX_STEPS = 0` the notebook
stops here, which is how the 6o2h breathing study was run.

In [ ]:
if WELD_TEST and INTERNAL.any():
    WELD = {}
    for o in np.flatnonzero(INTERNAL):
        chi_tr, chi_te = chi2(model, x, TRAIN), chi2(model, x, TEST)
        sp = [d for d in spectral(model, x, np.zeros(model.p)) if d['o'] == o]
        nC, nZ, nF = model.counts(o)
        print(f"internal orbit {ORB[o]} after the start: {nC} directions at infinite stiffness, {nZ} at zero, "
              f"{nF} finite; finite log-eigenvalues vs the prior {[round(d['sigma'], 2) for d in sp]}")
        _, J = resid_jac(model, x, TRAIN); g, V = np.linalg.eigh(J.T @ J)
        L = model.lay[o]; share = (V[L['off']:L['off'] + L['n']]**2).sum(0)
        ranks = [k for k in range(len(g)) if share[k] > 0.5]
        print(f"   Fisher spectrum ({len(g)} directions, g from {g[0]:.1e} to {g[-1]:.1e}): directions mostly in "
              f"this orbit rank {ranks[:8]} (0 = sloppiest)")
        for d in sp:
            for kind in ('C', 'Z'):
                try:
                    m2, x2 = apply_limit(model, x, o, d['j'], kind); jump = chi2(m2, x2, TRAIN) - chi_tr
                except np.linalg.LinAlgError:
                    jump = np.inf
                print(f"   eigenvalue {d['sigma']:+.2f} -> {'infinite' if kind == 'C' else 'zero'} stiffness: "
                      f"chi^2 change without refit {jump:+.3f}")
        mw, xw = model, x
        while mw.lay[o]['nF']:
            mw, xw = apply_limit(mw, xw, o, mw.lay[o]['nF'] - 1, 'C')
        jump = chi2(mw, xw, TRAIN) - chi_tr
        xw = fit_model(mw, xw)
        dchi, dp = chi2(mw, xw, TRAIN) - chi_tr, model.p - mw.p
        WELD[ORB[o]] = dict(jump=jump, dchi2=dchi, removed=dp, chi2_test=chi2(mw, xw, TEST), chi2_test_before=chi_te)
        print(f"   FULL WELD: {model.p} -> {mw.p} parameters; chi^2 change before refit {jump:+.2f}, after refit "
              f"{dchi:+.2f} (train {chi_tr:.1f} -> {chi_tr + dchi:.1f}, held out {chi_te:.1f} -> {chi2(mw, xw, TEST):.1f}); "
              f"Delta AIC {dchi - 2*dp:+.1f} (negative: the welded model is preferred)")
else:
    print("no internal orbits, or WELD_TEST = False")

## 6. The MBAM loop

In [ ]:
# --- The MBAM loop -------------------------------------------------------------------------------------
def _label_limit(o, kind, s):
    return f"{ORB[o]}{'*' if INTERNAL[o] else ''} {'C (k->inf)' if kind == 'C' else 'Z (k->0)'} at sigma {s:+.1f}"

for step in range(1, MBAM_MAX_STEPS + 1):
    if MBAM_TIME_BUDGET and time.time() - _t0 > MBAM_TIME_BUDGET:
        print("time budget reached"); break
    if model.p == 0:
        break
    ts = time.time()
    chi0 = chi2(model, x, TRAIN)
    dirs, gspec = start_directions(model, x, TRAIN)
    found = []
    # direct test of a nearly invisible direction
    for g, v, fam in dirs:
        if g < G_DIRECT:
            sp = spectral(model, x, v)
            if not sp:
                continue
            d = max(sp, key=lambda d: d['share'])
            r_x = resid(model, x, TRAIN)
            for kind in (['C'] if d['sigma'] > 1 else ['Z'] if d['sigma'] < -1 else ['C', 'Z']):
                m2, x2 = apply_limit(model, x, d['o'], d['j'], kind)
                try:
                    jump = float(np.sum((resid(m2, x2, TRAIN) - r_x)**2))
                except np.linalg.LinAlgError:
                    continue
                if jump < LIMIT_TOL:
                    found.append(dict(found=True, tau=0.0, limit=dict(d, sigma=d['sigma'] if kind == 'C' else -abs(d['sigma'])),
                                      kind=kind, model=m2, x=x2, jump=jump, how='direct', g=g))
                    break
            if found:
                break
    if not found:
        for g, v, fam in dirs:
            tau_cap = TAU_MAX
            for sgn in (1, -1):
                res = geodesic(model, x, sgn*v, TRAIN, tau_cap)
                if res['found']:
                    res.update(how='geodesic', g=g, fam=fam, kind='C' if res['limit']['sigma'] > 0 else 'Z')
                    found.append(res); tau_cap = min(tau_cap, res['tau'])
                    if BOUNDARY_CHOICE == 'closest':
                        break
    if not found:
        print(f"step {step}: no boundary within the budgets (sloppiest g = {gspec[0]:.2e}); MBAM stops here")
        break
    # refit each distinct boundary and choose
    uniq = {}
    for f in found:
        uniq.setdefault((f['limit']['o'], f['kind'], round(f['limit']['sigma'], 1)), f)
    cands = []
    for f in (uniq.values() if BOUNDARY_CHOICE == 'cheapest' else [min(found, key=lambda f: f['tau'])]):
        x2 = fit_model(f['model'], f['x'])
        cands.append((chi2(f['model'], x2, TRAIN), f, x2))
    c2, f, x = min(cands, key=lambda c: c[0])
    model = f['model']
    lim = [(f['limit']['o'], f['kind'], f['limit']['sigma'])]
    model, x, more = fit_driven(model, x)
    rec = dict(step=step, action=f['how'], limits=lim + more, p=model.p, chi2=chi2(model, x, TRAIN),
               chi2_test=chi2(model, x, TEST), tau=f['tau'], g=f['g'], status={o: model.counts(o) for o in range(NO)},
               model=model, x=x, chi_before=chi0, n_found=len(uniq), seconds=time.time() - ts)
    records.append(rec)
    print(f"step {step:2d} [{f['how']}, {len(uniq)} boundary(ies), g = {f['g']:.1e}, tau = {f['tau']:.2f}]: "
          + "; ".join(_label_limit(*l) for l in rec['limits'])
          + f"  -> p = {model.p}, chi^2 {chi0:.1f} -> {rec['chi2']:.1f} (held out {rec['chi2_test']:.1f}), "
          f"{rec['seconds']:.0f} s")
    for o in np.flatnonzero(INTERNAL):
        print(f"         {describe(model, o)}")
print(f"\n{len(records) - 1} MBAM steps in {time.time() - _t0:.0f} s; {model.p} parameters left")

## 7. Where the internal springs went

In [ ]:
# --- Where did the internal springs go? -------------------------------------------------------------------
print(f"{'step':>4s} {'action':>10s} {'p':>5s} {'chi2 train':>11s} {'Δchi2':>8s} {'chi2 held out':>14s}  limits (*: internal orbit)")
prev = None
for r in records:
    dchi = r['chi2'] - prev if prev is not None else 0.0
    print(f"{r['step']:4d} {r['action']:>10s} {r['p']:5d} {r['chi2']:11.1f} {dchi:8.2f} {r['chi2_test']:14.1f}  "
          + "; ".join(_label_limit(*l) for l in r['limits']))
    prev = r['chi2']
for o in np.flatnonzero(INTERNAL):
    steps_C = [(r['step'], l[2]) for r in records for l in r['limits'] if l[0] == o and l[1] == 'C']
    steps_Z = [(r['step'], l[2]) for r in records for l in r['limits'] if l[0] == o and l[1] == 'Z']
    print(f"\ninternal orbit {ORB[o]}: {len(steps_C)} of 6 directions sent to infinite stiffness "
          f"(steps {[s for s, _ in steps_C]}), {len(steps_Z)} released (steps {[s for s, _ in steps_Z]})")
    lat = [r['step'] for r in records for l in r['limits'] if l[0] != o]
    print(f"   limits on lattice orbits in the same steps: {len(lat)} (steps {sorted(set(lat))})")

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
P_ = [r['p'] for r in records]
ax[0].plot(P_, [r['chi2'] for r in records], 'o-', label='training')
ax[0].plot(P_, [r['chi2_test'] for r in records], 's--', label='held out')
ax[0].invert_xaxis(); ax[0].set_xlabel('parameters'); ax[0].set_ylabel(r'$\chi^2$'); ax[0].legend(fontsize=8)
ax[0].set_title('Fit quality along the reduction')
for o in range(NO):
    nC = [r['status'][o][0] for r in records]; nZ = [r['status'][o][1] for r in records]
    if max(nC) + max(nZ) == 0:
        continue
    st_ = '-' if INTERNAL[o] else ':'
    ax[1].step([r['step'] for r in records], nC, st_, where='post', lw=2.5 if INTERNAL[o] else 1.2,
               label=f"{ORB[o]}{' (internal)' if INTERNAL[o] else ''}: C")
    if max(nZ):
        ax[1].step([r['step'] for r in records], nZ, st_, where='post', lw=1, alpha=0.6, label=f"{ORB[o]}: Z")
ax[1].set_xlabel('MBAM step'); ax[1].set_ylabel('directions at the limit'); ax[1].set_ylim(-0.2, 6.2)
ax[1].set_title('Constrained (C) and released (Z) directions per orbit'); ax[1].legend(fontsize=7)
_, _J = resid_jac(records[0]['model'], records[0]['x'], TRAIN)
_g = np.linalg.eigvalsh(_J.T @ _J)[::-1]
ax[2].semilogy(np.arange(1, len(_g) + 1), np.maximum(_g, 1e-16), 'o', ms=3)
ax[2].axhline(1, color='r', ls='--', lw=1); ax[2].set_xlabel('index'); ax[2].set_ylabel('g')
ax[2].set_title('Fisher spectrum at the start')
plt.tight_layout(); plt.savefig(OUT + 'summary.png', dpi=130); plt.show()